# 02 · toy Markov speculative decoder

고정된 2-token Markov 언어 모델에서 기준 샘플러와 speculative sampler의 sequence 분포를 비교한다.

**학습 목표**: 첫 거절 이후 후보를 버리고 잔차 보정할 때 기준·추측 decoder의 경험적 sequence 분포가 가까워짐을 검증한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `random`, `collections`만 사용하며 외부 패키지는 없다.

In [ ]:
# 전이 확률을 중첩 dict로 두면 현재 token별 p·q를 같은 문법으로 조회할 수 있다.
import random
from collections import Counter

P = {'^': {'A': .65, 'B': .35}, 'A': {'A': .70, 'B': .30}, 'B': {'A': .25, 'B': .75}}
Q = {'^': {'A': .55, 'B': .45}, 'A': {'A': .60, 'B': .40}, 'B': {'A': .35, 'B': .65}}

def draw(dist, rng):
    u, acc = rng.random(), 0.0
    for token, probability in dist.items():
        acc += probability
        if u <= acc:
            return token
    return next(reversed(dist))

def target_sample(length, rng):
    out = []
    while len(out) < length:
        out.append(draw(P[out[-1] if out else '^'], rng))
    return ''.join(out)

def corrected_dist(p, q):
    raw = {x: max(0.0, p[x] - q[x]) for x in p}
    total = sum(raw.values())
    return {x: value / total for x, value in raw.items()}

def speculative_sample(length, gamma, rng):
    out = []
    while len(out) < length:
        draft, states = [], []
        state = out[-1] if out else '^'
        for _ in range(min(gamma, length - len(out))):
            states.append(state)
            token = draw(Q[state], rng)
            draft.append(token)
            state = token
        rejected = False
        for token, state in zip(draft, states):
            p, q = P[state], Q[state]
            if rng.random() <= min(1.0, p[token] / q[token]):
                out.append(token)
            else:
                out.append(draw(corrected_dist(p, q), rng))
                rejected = True
                break
            if len(out) == length:
                break
        if not rejected and len(out) < length:
            out.append(draw(P[out[-1]], rng))
    return ''.join(out[:length])


In [ ]:
n, length = 30_000, 4
baseline_rng, spec_rng = random.Random(11), random.Random(29)
baseline = Counter(target_sample(length, baseline_rng) for _ in range(n))
speculative = Counter(speculative_sample(length, 2, spec_rng) for _ in range(n))
keys = set(baseline) | set(speculative)
tv = 0.5 * sum(abs(baseline[k] / n - speculative[k] / n) for k in keys)
print('total variation (Monte Carlo):', round(tv, 4))
for key in sorted(keys)[:8]:
    print(key, round(baseline[key] / n, 4), round(speculative[key] / n, 4))
assert tv < 0.03


유한 표본 때문에 빈도는 정확히 같지 않지만 total variation이 작다. 실제 회귀 테스트는 더 많은 sample, 여러 seed, EOS와 production sampling transform을 포함해야 한다.